# Recomendador neural com PyTorch

## Objetivo

Treinar um modelo de recomendação personalizado com embeddings de usuários e filmes. O experimento combina dois objetivos:

- regressão explícita, que aproxima o rating observado com MSE;
- ranking pairwise, que coloca filmes relevantes acima de filmes não observados.

A validação escolhe o equilíbrio entre esses objetivos. O teste permanece fechado até a configuração final ser definida.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Não foi possível localizar data/raw.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data.loaders import load_movies, load_ratings
from data.splitting import temporal_leave_two_out
from evaluation.metrics import evaluate_top_k, mae, rmse
from models.factory import create_model
from models.pytorch_recommender import NeuralRecommenderConfig

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

TOP_K = 10
RELEVANCE_THRESHOLD = 4.0
RANDOM_STATE = 42

## 1. Dados e split temporal

A última interação de cada usuário representa o teste, a penúltima representa a validação e todo o histórico anterior forma o treino. O modelo nunca usa um evento futuro para aprender uma previsão do passado.

In [ ]:
raw_data_dir = PROJECT_ROOT / "data" / "raw"
ratings = load_ratings(raw_data_dir)
movies = load_movies(raw_data_dir)
train, validation, test = temporal_leave_two_out(ratings)

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "users": [
            train["user_id"].nunique(),
            validation["user_id"].nunique(),
            test["user_id"].nunique(),
        ],
        "movies": [
            train["movie_id"].nunique(),
            validation["movie_id"].nunique(),
            test["movie_id"].nunique(),
        ],
    }
)

## 2. Arquitetura e conexão com o feedback

Cada ID é convertido para um índice interno e consultado em uma tabela de embeddings. Para cada par, a rede concatena três vetores:

`embedding do usuário | embedding do filme | produto usuário × filme`

A MLP transforma esses sinais em um rating entre 0,5 e 5,0. A função de perda é híbrida:

`perda total = MSE dos ratings + ranking_weight × perda pairwise`

Ratings a partir de 4,0 são feedback positivo. Para cada positivo do batch, amostramos um filme que o usuário não avaliou e penalizamos a rede quando o negativo recebe score maior. Portanto, feedback explícito alimenta a regressão e também define quais exemplos participam do ranking.

In [ ]:
def evaluate_model(name, model, fit_data, holdout):
    started_at = perf_counter()
    fitted_model = model.fit(fit_data)
    train_seconds = perf_counter() - started_at
    predictions = fitted_model.predict_pairs(holdout)
    ranking, user_metrics, recommendations = evaluate_top_k(
        holdout,
        fitted_model.recommend,
        fitted_model.catalog_size,
        k=TOP_K,
        relevance_threshold=RELEVANCE_THRESHOLD,
    )
    actual = holdout["rating"].to_numpy(float)
    metrics = {
        "model": name,
        "rmse": rmse(actual, predictions),
        "mae": mae(actual, predictions),
        "precision@10": ranking["precision@10"],
        "recall@10": ranking["recall@10"],
        "ndcg@10": ranking["ndcg@10"],
        "hit_rate@10": ranking["hit_rate@10"],
        "coverage@10": ranking["catalog_coverage@10"],
        "train_seconds": train_seconds,
    }
    return fitted_model, metrics, user_metrics, recommendations

## 3. Experimento controlado na validação

Somente `ranking_weight` muda entre os candidatos. Isso isola o efeito do objetivo de ranking:

- 0,2 prioriza previsão de ratings;
- 1,0 busca equilíbrio;
- 2,0 prioriza a ordem da lista.

NDCG@10 é o critério de seleção porque o produto entrega uma lista ordenada. RMSE funciona como desempate.

In [ ]:
common_config = {
    "embedding_dim": 16,
    "hidden_dims": (32,),
    "learning_rate": 0.001,
    "weight_decay": 1e-5,
    "batch_size": 2048,
    "epochs": 8,
    "random_state": RANDOM_STATE,
}
candidate_configs = {
    "pytorch_rating_focused": NeuralRecommenderConfig(
        **common_config, ranking_weight=0.2
    ),
    "pytorch_balanced": NeuralRecommenderConfig(
        **common_config, ranking_weight=1.0
    ),
    "pytorch_ranking_focused": NeuralRecommenderConfig(
        **common_config, ranking_weight=2.0
    ),
}
candidate_configs

In [ ]:
validation_rows = []
validation_models = {}
for name, config in candidate_configs.items():
    model = create_model("pytorch", config=config)
    fitted, metrics, _, _ = evaluate_model(name, model, train, validation)
    validation_models[name] = fitted
    validation_rows.append(metrics)

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["ndcg@10", "rmse"], ascending=[False, True]
)
validation_results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, model in validation_models.items():
    epochs = range(1, len(model.history.train_rmse) + 1)
    ax.plot(epochs, model.history.train_rmse, marker="o", label=name)
ax.set(
    xlabel="Época",
    ylabel="RMSE de treino",
    title="Convergência das configurações PyTorch",
)
ax.legend()
plt.show()

## 4. Comparação justa com Scikit-Learn

O melhor baseline Scikit-Learn é treinado novamente no mesmo treino e avaliado na mesma validação. Complexidade não garante vitória: o objetivo é medir onde a rede melhora e onde ainda precisa evoluir.

In [ ]:
sklearn_model = create_model(
    "sklearn_bias", alpha=1e-4, random_state=RANDOM_STATE
)
_, sklearn_metrics, _, _ = evaluate_model(
    "sklearn_bias_alpha_1e-4", sklearn_model, train, validation
)
comparison = pd.concat(
    [validation_results, pd.DataFrame([sklearn_metrics])], ignore_index=True
).sort_values(["ndcg@10", "rmse"], ascending=[False, True])
comparison

In [ ]:
best_row = validation_results.sort_values(
    ["ndcg@10", "rmse"], ascending=[False, True]
).iloc[0]
best_name = best_row["model"]
best_config = candidate_configs[best_name]

pd.Series(
    {
        "selected_model": best_name,
        "selection_metric": "ndcg@10 (rmse como desempate)",
        "ranking_weight": best_config.ranking_weight,
    },
    name="selection",
)

## 5. Treino final e teste

Após a seleção, treino e validação são unidos. A rede é reinicializada e treinada do zero com mais histórico. O teste é usado uma única vez para estimar a generalização final.

In [ ]:
train_validation = pd.concat([train, validation], ignore_index=True)
final_model = create_model("pytorch", config=best_config)
final_model, test_metrics, test_user_metrics, test_recommendations = (
    evaluate_model(best_name, final_model, train_validation, test)
)
pd.Series(test_metrics, name="test")

## 6. Inspeção das listas

A inspeção traduz IDs para títulos e verifica que os filmes usados no treino final não reaparecem nas recomendações. Listas diferentes entre usuários são uma evidência simples de personalização.

In [ ]:
sample_users = test_user_metrics["user_id"].head(3).tolist()
sample_lists = {}
for user_id in sample_users:
    movie_ids = test_recommendations[int(user_id)]
    sample_lists[int(user_id)] = tuple(movie_ids)
    seen = set(
        train_validation.loc[train_validation["user_id"] == user_id, "movie_id"]
    )
    assert not set(movie_ids) & seen
    ranked = pd.DataFrame(
        {"rank": range(1, len(movie_ids) + 1), "movie_id": movie_ids}
    ).merge(movies, on="movie_id", how="left")
    print(f"\nUsuário {user_id}")
    display(ranked[["rank", "title", "genres"]])

print(f"Listas distintas na amostra: {len(set(sample_lists.values()))}")

## Conclusões

- Embeddings substituem IDs esparsos por vetores densos aprendidos.
- O produto dos embeddings representa compatibilidade usuário-filme; a MLP aprende relações não lineares.
- MSE e ranking pairwise resolvem problemas diferentes, por isso `ranking_weight` produz um trade-off mensurável.
- A seed controla inicialização, embaralhamento e amostragem negativa, permitindo reproduzir o experimento.
- IDs desconhecidos usam a média global, e filmes já vistos são filtrados antes do Top-K.
- O teste não participa de treinamento nem seleção. A próxima etapa de engenharia é persistir artefatos e rastrear experimentos com DVC e MLflow.